In [3]:
import pandas as pd

data = pd.read_csv('wine_quality_merged.csv')
df = pd.DataFrame(data)

#количество строк столбцов
num_rows, num_columns = df.shape
print(f"Размер датасета: {num_rows} строк, {num_columns} признаков")

#типы данных
print(f'Типы данных:\n {df.dtypes}')

#проверка наличия пропусков
print("\nКоличество пропусков по столбцам:")
print(df.isnull().sum())

#распределение классов
print("Сколько раз встречается каждый класс: ")
print(df['type'].value_counts())
print("Процентное соотношение: ")
print((df['type'].value_counts(normalize=True) * 100).astype(str) + "%")

#возможные проблемы в данных
#проверка на повторяющийся значения
duplicate = df.duplicated().sum()
print(f'Количество повторяющихся значений: {duplicate}')
df = df.drop_duplicates()
duplicate = df.duplicated().sum()
print(f'Количество повторяющихся значений: {duplicate}') #проверка



Размер датасета: 6497 строк, 13 признаков
Типы данных:
 fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
type                        str
dtype: object

Количество пропусков по столбцам:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
type                    0
dtype: int64
Сколько раз встречается каждый класс: 
type
white    4898
red      1599
Name: count, dtype: int64
Процентное соотношение: 
type


In [4]:
#Статистика для числовых признаков
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("\nМинимальные и максимальные значения числовых признаков:")
print(df[numerical_cols].agg(['min','max']))
print(df.describe())


Минимальные и максимальные значения числовых признаков:
     fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
min            3.8              0.08         0.00             0.6      0.009   
max           15.9              1.58         1.66            65.8      0.611   

     free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
min                  1.0                   6.0  0.98711  2.72       0.22   
max                289.0                 440.0  1.03898  4.01       2.00   

     alcohol  quality  
min      8.0        3  
max     14.9        9  
       fixed acidity  volatile acidity  citric acid  residual sugar  \
count    5320.000000       5320.000000  5320.000000     5320.000000   
mean        7.215179          0.344130     0.318494        5.048477   
std         1.319671          0.168248     0.147157        4.500180   
min         3.800000          0.080000     0.000000        0.600000   
25%         6.400000          0.230000     0.2

In [13]:
#Подготовка данных
#кодирование категориальных признаков (если есть)
df = pd.get_dummies(df)

#масштабирование признаков
num_columns = df.select_dtypes(include=['int64', 'float64']).columns
from sklearn.preprocessing import RobustScaler
transformer = RobustScaler()
df[num_columns] = transformer.fit_transform(df[num_columns])
df.head()

#разделение на train / test
from sklearn.model_selection import train_test_split

X = df.drop(['type_white'], axis=1)
y = df['type_white']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.4, 
    random_state=42
    )


'''
#почему масштабирование важно для KNN
Так как алгоритм KNN ищет ближайших соседей по растоянию. То признаки с большими значенисм будут доминировать

#почему нельзя подбирать параметры на тестовой выборке
Потому что тестовая выборка используется для оценки модели. И лучше проверять модель на данных которые не участвовали в обучении
'''

'\n#почему масштабирование важно для KNN\nТак как алгоритм KNN ищет ближайших соседей по растоянию. То признаки с большими значенисм будут доминировать\n\n#почему нельзя подбирать параметры на тестовой выборке\nПотому что тестовая выборка используется для оценки модели. И лучше проверять модель на данных которые не участвовали в обучении\n'

In [ ]:
#обучение KNN
#uniform для всех соседей одинаковый вес distance чем ближе тем больше вес
#n_neighbors количество ближайжих соседей которое используется для предсказания
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
for k in [1, 3, 5, 7, 9]:
    for i in ['uniform', 'distance']:
        for metrik in ['euclidean', 'manhattan', 'minkowski']:
            model = KNeighborsClassifier(
            n_neighbors=k,
            weights=i 
            )
            model.fit(X_train, y_train)
    #предсказание
            pred = model.predict(X_test)
            print(f'n_neighbors {k} {i} {metrik}: {accuracy_score(y_test, pred)}')
#конкретно для моих данных на точность повлияло количество соседей и тип весов при
#n_neighbors 1 uniform n_neighbors 1 distance был достигнут один наивысший результат предсказания

n_neighbors 1 uniform euclidean: 0.9981203007518797
n_neighbors 1 uniform manhattan: 0.9981203007518797
n_neighbors 1 uniform minkowski: 0.9981203007518797
n_neighbors 1 distance euclidean: 0.9981203007518797
n_neighbors 1 distance manhattan: 0.9981203007518797
n_neighbors 1 distance minkowski: 0.9981203007518797
n_neighbors 3 uniform euclidean: 0.9976503759398496
n_neighbors 3 uniform manhattan: 0.9976503759398496
n_neighbors 3 uniform minkowski: 0.9976503759398496
n_neighbors 3 distance euclidean: 0.9981203007518797
n_neighbors 3 distance manhattan: 0.9981203007518797
n_neighbors 3 distance minkowski: 0.9981203007518797
n_neighbors 5 uniform euclidean: 0.9967105263157895
n_neighbors 5 uniform manhattan: 0.9967105263157895
n_neighbors 5 uniform minkowski: 0.9967105263157895
n_neighbors 5 distance euclidean: 0.9976503759398496
n_neighbors 5 distance manhattan: 0.9976503759398496
n_neighbors 5 distance minkowski: 0.9976503759398496
n_neighbors 7 uniform euclidean: 0.9971804511278195
n_n

In [ ]:
#5